In [ ]:
pip install requests beautifulsoup4 lxml langdetect python-dateutil openpyxl

**Step 1: Load the Excel file**

In [ ]:
import pandas as pd

# Change path if needed
INPUT_PATH = "/content/REST_Media_Backlinks (1).xlsx"

df = pd.read_excel(INPUT_PATH)
print("Loaded rows:", len(df))
display(df.head())

Loaded rows: 28


,Page ascore,Source title,Source url,Target url,Anchor,External links,Internal links,Nofollow,Sponsored,Ugc,Text,Frame,Form,Image,Sitewide,First seen,Last seen,New link,Lost link
0,0,5000 domain name ideas for word - rest | DomenSky,https://domains.bugcrew.net/domains/rest,https://restmedia.st/,.io,24995,17,False,False,False,True,False,False,False,False,2025-08-22,2025-09-26,False,False
1,0,9 letter domain name ideas for word - rest | D...,https://domains.bugcrew.net/domains/rest/9,https://restmedia.st/,.io,4570,4,False,False,False,True,False,False,False,False,2025-08-26,2025-09-26,False,False
2,0,SYNLOGOS,https://dstormer6em3i4km.onion.pw/sources,https://restmedia.st/feed/,âŒ”,717,12,False,False,False,False,False,False,True,False,2025-06-30,2025-09-27,False,False
3,0,SYNLOGOS,https://dstormer6em3i4km.onion.pw/sources,https://restmedia.st/,https://restmedia.io,717,12,False,False,False,True,False,False,False,False,2025-06-30,2025-09-27,False,False
4,0,SYNLOGOS,https://elstormer6vrre53.onion.pw/sources,https://restmedia.st/feed/,âŒ”,717,12,False,False,False,False,False,False,True,False,2025-06-30,2025-09-21,False,False


In [ ]:
columns_to_drop = [
    'Anchor',
    'External links',
    'Internal links',
    'Nofollow',
    'Sponsored',
    'Ugc',
    'Text',
    'Frame',
    'Form',
    'Image',
    'Sitewide',
    'First seen',
    'Last seen',
    'New link',
    'Lost link'
]

df = df.drop(columns=columns_to_drop, errors='ignore')

print("Columns after dropping:", df.columns.tolist())
display(df.head())

Columns after dropping: ['Page ascore', 'Source title', 'Source url', 'Target url']


,Page ascore,Source title,Source url,Target url
0,0,5000 domain name ideas for word - rest | DomenSky,https://domains.bugcrew.net/domains/rest,https://restmedia.st/
1,0,9 letter domain name ideas for word - rest | D...,https://domains.bugcrew.net/domains/rest/9,https://restmedia.st/
2,0,SYNLOGOS,https://dstormer6em3i4km.onion.pw/sources,https://restmedia.st/feed/
3,0,SYNLOGOS,https://dstormer6em3i4km.onion.pw/sources,https://restmedia.st/
4,0,SYNLOGOS,https://elstormer6vrre53.onion.pw/sources,https://restmedia.st/feed/


**Step 2: Import libraries and silence XML-as-HTML warning**

In [ ]:
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
import requests
from dateutil import parser
from langdetect import detect

# Silence BeautifulSoup's XMLParsedAsHTMLWarning (optional, but tidy)
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

**Step 3: Fetch URL content (with basic error handling)**

In [ ]:
def fetch_url(url, timeout=12):
    try:
        headers = {"User-Agent": "Mozilla/5.0 (compatible; ExtractionBot/1.0)"}
        r = requests.get(url, timeout=timeout, headers=headers)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f"[FETCH ERROR] {url} -> {e}")
        return None

# quick test (uncomment one URL to test)
# print(fetch_url("https://restmedia.st/")[:500])

In [ ]:
def is_xml_content(text):
    if not text:
        return False
    s = text.lstrip()[:100].lower()
    return s.startswith("<?xml") or s.startswith("<rss") or s.startswith("<feed") or "<rss" in s[:200] or "<feed" in s[:200]

In [ ]:
def parse_title(soup):
    # HTML <title> or XML <title> inside items/entry
    title_tag = soup.find("title")
    if title_tag and title_tag.get_text(strip=True):
        return title_tag.get_text(strip=True)
    # fallback: meta og:title
    meta = soup.find("meta", property="og:title") or soup.find("meta", attrs={"name":"twitter:title"})
    if meta and meta.get("content"):
        return meta["content"].strip()
    return None

In [ ]:
def parse_text(soup, xml_mode=False, max_chars=20000):
    text_pieces = []
    if not xml_mode:
        # HTML: get paragraphs and article tags
        for p in soup.find_all("p"):
            t = p.get_text(" ", strip=True)
            if t:
                text_pieces.append(t)
        # also try article tag if paragraphs empty
        if not text_pieces:
            article = soup.find("article")
            if article:
                text_pieces.append(article.get_text(" ", strip=True))
    else:
        # XML: look for item/entry descriptions or content:encoded
        for tagname in ("content:encoded", "encoded", "description", "summary", "content"):
            for tag in soup.find_all(tagname):
                t = tag.get_text(" ", strip=True)
                if t:
                    text_pieces.append(t)
        # sometimes title+link per item is the only content; include item titles
        if not text_pieces:
            for item in soup.find_all(["item","entry"]):
                t = ""
                t += (item.find("title").get_text(" ", strip=True) if item.find("title") else "")
                desc = item.find("description") or item.find("summary")
                if desc:
                    t += " " + desc.get_text(" ", strip=True)
                if t.strip():
                    text_pieces.append(t.strip())

    text = " ".join(text_pieces).strip()
    if not text:
        return None
    return text[:max_chars]

In [ ]:
def parse_date(soup):
    # Common date tags
    candidates = []
    # HTML5 time tag
    for t in soup.find_all("time"):
        if t.has_attr("datetime"):
            candidates.append(t["datetime"])
        else:
            candidates.append(t.get_text(" ", strip=True))
    # RSS/Atom
    for name in ("pubDate","published","updated","dc:date","date"):
        for tag in soup.find_all(name):
            candidates.append(tag.get_text(" ", strip=True))
    # Meta tags
    meta_names = [
        ("meta", {"property":"article:published_time"}),
        ("meta", {"name":"pubdate"}),
        ("meta", {"name":"publishdate"}),
        ("meta", {"name":"publication_date"})
    ]
    for tagname, attrs in meta_names:
        tag = soup.find(tagname, attrs=attrs)
        if tag and tag.get("content"):
            candidates.append(tag["content"])

    # Try to parse candidates with dateutil.parser
    for c in candidates:
        if not c or len(c.strip())<4:
            continue
        try:
            dt = parser.parse(c, fuzzy=True)
            return dt.isoformat()
        except Exception:
            continue
    return None

In [ ]:
def detect_language(text):
    try:
        if not text or len(text.strip()) < 30:
            return None
        return detect(text)
    except Exception:
        return None

In [ ]:
def extract_from_url(url):
    raw = fetch_url(url)
    if raw is None:
        return {"title": None, "text": None, "date": None, "language": None}

    xml_mode = is_xml_content(raw)
    parser_type = "xml" if xml_mode else "html.parser"
    soup = BeautifulSoup(raw, parser_type)

    title = parse_title(soup)
    text = parse_text(soup, xml_mode=xml_mode)
    date = parse_date(soup)
    language = detect_language(text)

    return {"title": title, "text": text, "date": date, "language": language}

In [ ]:
from tqdm import tqdm

results = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    source = row.get("Source url") or row.get("Source_url") or row.get("SourceUrl")
    target = row.get("Target url") or row.get("Target_url") or row.get("TargetUrl")

    res = {"row_index": idx, "source_url": source, "target_url": target}

    if source and isinstance(source, str) and source.strip():
        out = extract_from_url(source.strip())
        res.update({
            "source_title": out["title"],
            "source_text": out["text"],
            "source_date": out["date"],
            "source_language": out["language"]
        })
    else:
        res.update({"source_title":None,"source_text":None,"source_date":None,"source_language":None})

    if target and isinstance(target, str) and target.strip():
        out = extract_from_url(target.strip())
        res.update({
            "target_title": out["title"],
            "target_text": out["text"],
            "target_date": out["date"],
            "target_language": out["language"]
        })
    else:
        res.update({"target_title":None,"target_text":None,"target_date":None,"target_language":None})

    results.append(res)

final_df = pd.DataFrame(results)

# Arrange columns in a proper order
cols = [
    "row_index",
    "source_url", "source_title", "source_text", "source_date", "source_language",
    "target_url", "target_title", "target_text", "target_date", "target_language"
]
final_df = final_df[cols]

# Save
OUT_PATH = "extracted_results_separate_steps.xlsx"
final_df.to_excel(OUT_PATH, index=False)
print("Saved extraction to:", OUT_PATH)

  0%|          | 0/28 [00:00<?, ?it/s]

[FETCH ERROR] https://domains.bugcrew.net/domains/rest -> HTTPSConnectionPool(host='domains.bugcrew.net', port=443): Max retries exceeded with url: /domains/rest (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7a80a3482270>: Failed to resolve 'domains.bugcrew.net' ([Errno -2] Name or service not known)"))


  4%|▎         | 1/28 [00:03<01:22,  3.05s/it]

[FETCH ERROR] https://domains.bugcrew.net/domains/rest/9 -> HTTPSConnectionPool(host='domains.bugcrew.net', port=443): Max retries exceeded with url: /domains/rest/9 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7a80a1e84980>: Failed to resolve 'domains.bugcrew.net' ([Errno -2] Name or service not known)"))


  7%|▋         | 2/28 [00:04<01:00,  2.34s/it]

...

[FETCH ERROR] https://vaseljenska.net/2025/06/28/oberer-angriff-2-0-ultimatumi-srbiji-na-vidovdan-i-dalje-stizu/ -> 403 Client Error: Forbidden for url: https://vaseljenska.net/2025/06/28/oberer-angriff-2-0-ultimatumi-srbiji-na-vidovdan-i-dalje-stizu/


 93%|█████████▎| 26/28 [00:45<00:03,  1.57s/it]

[FETCH ERROR] https://www.marx21.it/internazionale/e-trasparenza-o-censura-la-battaglia-della-moldavia-contro-la-disinformazione/ -> 403 Client Error: Forbidden for url: https://www.marx21.it/internazionale/e-trasparenza-o-censura-la-battaglia-della-moldavia-contro-la-disinformazione/


100%|██████████| 28/28 [00:51<00:00,  1.84s/it]

Saved extraction to: extracted_results_separate_steps.xlsx
